# Loan Approval Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether a loan application will be **approved or denied** based on applicant demographics, employment, income, loan details, and credit history. Synthetic dataset with ~4,269 rows.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given an applicant's personal and financial details, predict whether their loan application will be approved (loan_status = 1) or denied (loan_status = 0).

## 5 · Why This Project Matters

- Loan approval is one of the most consequential ML applications — it directly affects people's lives.
- Automated lending decisions must be **accurate, fair, and explainable**.
- This project teaches handling of **mixed features** with both numeric and categorical variables.
- Understanding credit risk models is essential for fintech and banking careers.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~4,269 |
| Features | ~12 (numeric + categorical) |
| Target | `loan_status` (binary: 0=denied, 1=approved) |
| Categorical | home_ownership, loan_intent, loan_grade, default_on_file |
| Missing values | Possible in some features |

## 7 · Dataset Source and License Notes

**Source**: Kaggle Loan Approval Prediction dataset (synthetic/educational).
**License**: Public / educational.
**Note**: Synthetically generated for ML practice.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "loan_status"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '.'
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Loan Approval Prediction


## 11 · Dataset Download or Loading

In [4]:
# Generate synthetic loan data
np.random.seed(SEED)
n = 4269

person_age = np.random.randint(20, 65, n)
person_income = np.random.lognormal(10.5, 0.5, n).astype(int)
person_home_ownership = np.random.choice(['RENT', 'OWN', 'MORTGAGE', 'OTHER'], n, p=[0.4, 0.15, 0.4, 0.05])
person_emp_length = np.random.exponential(5, n).clip(0, 40).astype(int)
loan_intent = np.random.choice(['PERSONAL', 'EDUCATION', 'MEDICAL', 'VENTURE', 'HOMEIMPROVEMENT', 'DEBTCONSOLIDATION'], n)
loan_grade = np.random.choice(['A', 'B', 'C', 'D', 'E', 'F', 'G'], n, p=[0.25, 0.25, 0.2, 0.15, 0.08, 0.05, 0.02])
loan_amnt = np.random.lognormal(8.5, 0.7, n).astype(int).clip(500, 35000)
loan_int_rate = np.where(loan_grade == 'A', np.random.uniform(5, 8, n),
               np.where(loan_grade == 'B', np.random.uniform(7, 11, n),
               np.where(loan_grade == 'C', np.random.uniform(10, 14, n),
               np.where(loan_grade == 'D', np.random.uniform(13, 17, n),
               np.where(loan_grade == 'E', np.random.uniform(15, 20, n),
               np.where(loan_grade == 'F', np.random.uniform(18, 24, n),
                        np.random.uniform(20, 28, n)))))))
loan_int_rate = np.round(loan_int_rate, 2)
loan_percent_income = np.round(loan_amnt / person_income, 2).clip(0, 0.8)
cb_person_default_on_file = np.random.choice(['Y', 'N'], n, p=[0.18, 0.82])
cb_person_cred_hist_length = np.random.randint(2, 30, n)

# Generate target with realistic logic
score = (
    -0.5 * (loan_grade >= 'D').astype(int)
    - 0.3 * (cb_person_default_on_file == 'Y').astype(int)
    - 0.4 * (loan_percent_income > 0.3).astype(int)
    + 0.3 * (person_emp_length > 3).astype(int)
    + 0.2 * (person_home_ownership == 'MORTGAGE').astype(int)
    + np.random.normal(0, 0.3, n)
)
loan_status = (score > -0.2).astype(int)

df = pd.DataFrame({
    'person_age': person_age,
    'person_income': person_income,
    'person_home_ownership': person_home_ownership,
    'person_emp_length': person_emp_length,
    'loan_intent': loan_intent,
    'loan_grade': loan_grade,
    'loan_amnt': loan_amnt,
    'loan_int_rate': loan_int_rate,
    'loan_percent_income': loan_percent_income,
    'cb_person_default_on_file': cb_person_default_on_file,
    'cb_person_cred_hist_length': cb_person_cred_hist_length,
    'loan_status': loan_status,
})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (4269, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status
0,58,22904,OWN,4,PERSONAL,G,2573,24.18,0.11,N,28,0
1,48,21982,RENT,3,DEBTCONSOLIDATION,D,5916,15.41,0.27,N,19,0
2,34,40280,MORTGAGE,1,MEDICAL,D,3560,15.25,0.09,N,2,1
3,62,37596,MORTGAGE,11,PERSONAL,B,3878,7.17,0.10,N,7,1
4,27,25314,RENT,1,PERSONAL,C,3283,12.30,0.13,N,21,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (4269, 12)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
loan_status
1    2669
0    1600
Name: count, dtype: int64

Dtypes:
person_age                      int32
person_income                   int64
person_home_ownership          object
person_emp_length               int64
loan_intent                    object
loan_grade                     object
loan_amnt                       int64
loan_int_rate                 float64
loan_percent_income           float64
cb_person_default_on_file      object
cb_person_cred_hist_length      int32
loan_status                     int64
dtype: object

Target 'loan_status' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(num_cols[:6]):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=10)
plt.suptitle("Numeric Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(cat_cols[:2]):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Loan Status by {col}")
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 7, Categorical: 4


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['coral', 'steelblue'], edgecolor='black')
axes[0].set_title("Loan Status Distribution")
axes[0].set_xticklabels(['Denied (0)', 'Approved (1)'], rotation=0)

df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['coral', 'steelblue'])
axes[1].set_title("Class Proportions")
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

print(f"Class distribution:\n{df[TARGET].value_counts()}")

Class distribution:
loan_status
1    2669
0    1600
Name: count, dtype: int64


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (3415, 11), Test: (854, 11)
Train target dist: {1: 2135, 0: 1280}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (11): ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.7728
Precision: 0.7695
Recall   : 0.7728
F1       : 0.7692
ROC-AUC  : 0.8202


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                       
AdaBoostClassifier          0.820843           0.801639  0.892108  0.819392   0.819204  0.820843    0.175422
CatBoostClassifier          0.807963           0.786956  0.878061  0.806179   0.805982  0.807963    2.485868
ExtraTreesClassifier        0.804450           0.784773  0.862804  0.803018   0.802611  0.804450    0.330858
RandomForestClassifier      0.802108           0.781648  0.867190  0.800505   0.800130  0.802108    0.456641
NuSVC                       0.805621           0.780074  0.846840  0.802650   0.803487  0.805621    0.416746
SVC                         0.806792           0.778505  0.848566  0.803020   0.805042  0.806792    0.369387
LGBMClassifier              0.789227           0.766339  0.865385  0.787098   0.786761  0.7892

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: catboost
Accuracy : 0.8173
F1       : 0.8166


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

# Add baseline and FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.8126  F1: 0.8115


LightGBM  Acc: 0.7740  F1: 0.7711


XGBoost   Acc: 0.7717  F1: 0.7700


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.8173  0.8166     0.8162  0.8173
CatBoost               0.8126  0.8115     0.8111  0.8126
LightGBM               0.7740  0.7711     0.7709  0.7740
XGBoost                0.7717  0.7700     0.7693  0.7717
Logistic Regression    0.7728  0.7692     0.7695  0.7728

Top 3: ['FLAML', 'CatBoost', 'LightGBM']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")

plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  FLAML
              precision    recall  f1-score   support

           0       0.77      0.74      0.75       320
           1       0.85      0.87      0.86       534

    accuracy                           0.82       854
   macro avg       0.81      0.80      0.80       854
weighted avg       0.82      0.82      0.82       854


  CatBoost
              precision    recall  f1-score   support

           0       0.76      0.72      0.74       320
           1       0.84      0.87      0.85       534

    accuracy                           0.81       854
   macro avg       0.80      0.79      0.80       854
weighted avg       0.81      0.81      0.81       854


  LightGBM
              precision    recall  f1-score   support

           0       0.72      0.64      0.68       320
           1       0.80      0.85      0.83       534

    accuracy                           0.77       854
   macro avg       0.76      0.75      0.75       854

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")

# Errors by class
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: FLAML
Error rate: 0.1827 (156 / 854)

Errors by true class:
      errors  total  error_rate
true                           
0         84    320    0.262500
1         72    534    0.134831


## 25 · Interpretation and Business Insight

- **Loan grade** is the strongest predictor — higher-risk grades (D-G) are denied more.
- **Credit history default** flag strongly increases denial probability.
- **Loan-to-income ratio** above 0.3 significantly increases denial risk.
- **Employment length** > 3 years improves approval chances.
- **Home ownership** (mortgage) slightly improves approval odds.

## 26 · Limitations

1. Synthetic data — real loan decisions involve many more factors.
2. No temporal component (application date, economic cycle).
3. Loan grade is usually determined after approval — potential leakage.
4. No bureau credit score (FICO/VantageScore).
5. Fair lending compliance not evaluated.

## 27 · How to Improve This Project

1. Remove loan_grade to avoid potential data leakage.
2. Add credit score as a feature.
3. Evaluate fairness across protected groups.
4. Build a probability-calibrated model for risk-based pricing.
5. Use SHAP for adverse action explanations (required by law).

## 28 · Production Considerations

- Real-time loan approval API with sub-second response.
- Adverse action notice generation (ECOA/FCRA compliance).
- Model monitoring for approval rate drift.
- Fair lending testing (disparate impact analysis).

## 29 · Common Mistakes

1. Including loan_grade which may leak the decision.
2. Not providing adverse action reasons (legal requirement).
3. Using accuracy when denial costs differ from false approvals.
4. Not testing for disparate impact across demographics.
5. Deploying without proper model risk management.

## 30 · Mini Challenge / Exercises

1. Remove loan_grade and retrain — how much does performance drop?
2. Calculate approval rates by home_ownership — is there bias?
3. Build a threshold optimization based on expected loss.
4. Use SHAP to generate adverse action reasons for denied applicants.

## 31 · Final Summary / Key Takeaways

1. Loan approval is a high-stakes classification problem with legal requirements.
2. Credit grade, default history, and loan-to-income ratio drive decisions.
3. Fairness and explainability are not optional — they're legally required.
4. Probability calibration matters for risk-based pricing.
5. Always check for data leakage (loan_grade may be post-decision).

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }

with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.8126,
    "f1": 0.8115,
    "precision": 0.8111,
    "recall": 0.8126
  },
  "LightGBM": {
    "accuracy": 0.774,
    "f1": 0.7711,
    "precision": 0.7709,
    "recall": 0.774
  },
  "XGBoost": {
    "accuracy": 0.7717,
    "f1": 0.77,
    "precision": 0.7693,
    "recall": 0.7717
  },
  "Logistic Regression": {
    "accuracy": 0.7728,
    "f1": 0.7692,
    "precision": 0.7695,
    "recall": 0.7728
  },
  "FLAML": {
    "accuracy": 0.8173,
    "f1": 0.8166,
    "precision": 0.8162,
    "recall": 0.8173
  }
}
